# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [13]:
# Load data with a fallback encoding for legacy characters in this file.
data_path = "data/AviationData.csv"
df = pd.read_csv(data_path, low_memory=False, encoding="latin1")

print("Initial shape:", df.shape)
print("\nSample rows:")
display(df.head())

print("\nColumn dtypes (first 15):")
display(df.dtypes.head(15))

print("\nMissing values (%), top 15 columns:")
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.head(15).round(2))

print("\nNumeric summary for key injury columns:")
key_num_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Number.of.Engines",
]
display(df[key_num_cols].describe(include="all").T)

Initial shape: (88889, 31)

Sample rows:


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980



Column dtypes (first 15):


Event.Id               str
Investigation.Type     str
Accident.Number        str
Event.Date             str
Location               str
Country                str
Latitude               str
Longitude              str
Airport.Code           str
Airport.Name           str
Injury.Severity        str
Aircraft.damage        str
Aircraft.Category      str
Registration.Number    str
Make                   str
dtype: object


Missing values (%), top 15 columns:


Schedule                  85.85
Air.carrier               81.27
FAR.Description           63.97
Aircraft.Category         63.68
Longitude                 61.33
Latitude                  61.32
Airport.Code              43.60
Airport.Name              40.71
Broad.phase.of.flight     30.56
Publication.Date          15.49
Total.Serious.Injuries    14.07
Total.Minor.Injuries      13.42
Total.Fatal.Injuries      12.83
Engine.Type                7.98
Report.Status              7.18
dtype: float64


Numeric summary for key injury columns:


,count,mean,std,min,25%,50%,75%,max
Total.Fatal.Injuries,77488.0,0.647855,5.485960,0.0,0.0,0.0,0.0,349.0
Total.Serious.Injuries,76379.0,0.279881,1.544084,0.0,0.0,0.0,0.0,161.0
Total.Minor.Injuries,76956.0,0.357061,2.235625,0.0,0.0,0.0,0.0,380.0
Total.Uninjured,82977.0,5.325440,27.913634,0.0,0.0,1.0,2.0,699.0
Number.of.Engines,82805.0,1.146585,0.446510,0.0,1.0,1.0,1.0,8.0


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [14]:
df_clean = df.copy()

# Standardize date and text fields used in filtering.
df_clean["Event.Date"] = pd.to_datetime(df_clean["Event.Date"], errors="coerce")
df_clean["Aircraft.Category"] = df_clean["Aircraft.Category"].astype("string").str.strip().str.lower()
df_clean["Amateur.Built"] = df_clean["Amateur.Built"].astype("string").str.strip().str.lower()
df_clean["Investigation.Type"] = df_clean["Investigation.Type"].astype("string").str.strip().str.lower()

rows_before = len(df_clean)

# Client scope filters.
df_clean = df_clean[df_clean["Event.Date"].dt.year >= 1983]
df_clean = df_clean[df_clean["Aircraft.Category"].eq("airplane")]
df_clean = df_clean[df_clean["Amateur.Built"].eq("no")]
df_clean = df_clean[df_clean["Investigation.Type"].eq("accident")]

# Keep rows with make/model available for downstream recommendations.
df_clean = df_clean.dropna(subset=["Make", "Model"])

rows_after = len(df_clean)
print(f"Rows before filtering: {rows_before:,}")
print(f"Rows after filtering:  {rows_after:,}")
print(f"Rows removed:          {rows_before - rows_after:,}")

display(df_clean[["Event.Date", "Aircraft.Category", "Amateur.Built", "Investigation.Type"]].head())

Rows before filtering: 88,889
Rows after filtering:  19,923
Rows removed:          68,966


,Event.Date,Aircraft.Category,Amateur.Built,Investigation.Type
4171,1983-03-20,airplane,no,accident
4285,1983-04-02,airplane,no,accident
5960,1983-08-21,airplane,no,accident
6669,1983-10-28,airplane,no,accident
6806,1983-11-13,airplane,no,accident


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [15]:
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
]

# Assumption: missing injury counts are treated as 0 for this event-level estimate.
for col in injury_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0)
    df_clean[col] = df_clean[col].clip(lower=0)

# Estimated people onboard from reported injury outcome columns.
df_clean["total_passengers_est"] = df_clean[injury_cols].sum(axis=1)
df_clean["fatal_serious_count"] = df_clean["Total.Fatal.Injuries"] + df_clean["Total.Serious.Injuries"]

df_clean["fatal_serious_fraction"] = np.where(
    df_clean["total_passengers_est"] > 0,
    df_clean["fatal_serious_count"] / df_clean["total_passengers_est"],
    np.nan,
)

# Keep events where estimated onboard count is known and positive.
df_clean = df_clean[df_clean["total_passengers_est"] > 0].copy()

display(
    df_clean[["total_passengers_est", "fatal_serious_count", "fatal_serious_fraction"]]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])
    .T
)
print("Rows after requiring total_passengers_est > 0:", f"{len(df_clean):,}")

,count,mean,std,min,25%,50%,75%,90%,99%,max
total_passengers_est,19662.0,6.189808,28.681509,1.0,1.0,2.0,2.0,4.0,165.0,576.0
fatal_serious_count,19662.0,0.996592,7.078648,0.0,0.0,0.0,1.0,2.0,7.0,295.0
fatal_serious_fraction,19662.0,0.295899,0.436685,0.0,0.0,0.0,1.0,1.0,1.0,1.0


Rows after requiring total_passengers_est > 0: 19,662


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [16]:
df_clean["Aircraft.damage"] = (
    df_clean["Aircraft.damage"]
    .astype("string")
    .str.strip()
    .str.title()
    .replace({"": pd.NA, "Nan": pd.NA, "N/A": pd.NA})
)

# Binary outcome used in risk comparison.
df_clean["destroyed_flag"] = (
    df_clean["Aircraft.damage"].eq("Destroyed").fillna(False).astype(int)
)

display(df_clean["Aircraft.damage"].value_counts(dropna=False).head(10))
print("Destroyed fraction (overall):", round(df_clean["destroyed_flag"].mean(), 4))

Aircraft.damage
Substantial    16794
Destroyed       2263
<NA>             407
Minor            134
Unknown           64
Name: count, dtype: Int64

Destroyed fraction (overall): 0.1151


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [17]:
# Standardize make names for consistent grouping.
df_clean["Make_Clean"] = (
    df_clean["Make"]
    .astype("string")
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

make_counts = df_clean["Make_Clean"].value_counts()
print("Unique makes before threshold:", make_counts.shape[0])

# Keep makes with enough observations for stable averages.
min_make_n = 50
valid_makes = make_counts[make_counts >= min_make_n].index
df_clean = df_clean[df_clean["Make_Clean"].isin(valid_makes)].copy()

print("Unique makes after threshold:", df_clean["Make_Clean"].nunique())
print("Rows after make threshold:", f"{len(df_clean):,}")
print("Top 15 makes by count:")
display(df_clean["Make_Clean"].value_counts().head(15))

Unique makes before threshold: 1044
Unique makes after threshold: 34
Rows after make threshold: 16,337
Top 15 makes by count:


Make_Clean
CESSNA                6959
PIPER                 3912
BEECH                 1382
BOEING                 485
MOONEY                 356
BELLANCA               217
AIR TRACTOR INC        217
MAULE                  215
CIRRUS DESIGN CORP     209
AIR TRACTOR            203
AERONCA                200
CHAMPION               157
GRUMMAN                146
LUSCOMBE               141
STINSON                129
Name: count, dtype: Int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [18]:
df_clean["Model_Clean"] = (
    df_clean["Model"]
    .astype("string")
    .fillna("UNKNOWN_MODEL")
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Model labels are not globally unique, so combine make+model as a stable plane type id.
df_clean["plane_type"] = df_clean["Make_Clean"] + " | " + df_clean["Model_Clean"]

models_with_multiple_makes = (
    df_clean.groupby("Model_Clean")["Make_Clean"].nunique().loc[lambda s: s > 1].shape[0]
)

print("Distinct model labels:", df_clean["Model_Clean"].nunique())
print("Distinct plane types (make+model):", df_clean["plane_type"].nunique())
print("Model labels appearing under >1 make:", models_with_multiple_makes)

display(df_clean[["Make_Clean", "Model_Clean", "plane_type"]].head())

Distinct model labels: 1814
Distinct plane types (make+model): 1916
Model labels appearing under >1 make: 94


,Make_Clean,Model_Clean,plane_type
4171,PIPER,PA-28-140,PIPER | PA-28-140
4285,DE HAVILLAND,DHC-6,DE HAVILLAND | DHC-6
6806,BEECH,C35,BEECH | C35
7084,CESSNA,180K,CESSNA | 180K
7708,BEECH,99,BEECH | 99


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [19]:
# Number of engines as nullable integer.
df_clean["Number.of.Engines"] = pd.to_numeric(df_clean["Number.of.Engines"], errors="coerce").astype("Int64")

# Standardize categorical factor fields used later in analysis.
def clean_category(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"": pd.NA, "UNK": pd.NA, "UNKNOWN": pd.NA, "N/A": pd.NA, "Nan": pd.NA})
    )

factor_cols = ["Engine.Type", "Weather.Condition", "Purpose.of.flight", "Broad.phase.of.flight"]
for col in factor_cols:
    if col in df_clean.columns:
        df_clean[col] = clean_category(df_clean[col])

# Normalize label case.
if "Engine.Type" in df_clean.columns:
    df_clean["Engine.Type"] = df_clean["Engine.Type"].str.title()
if "Purpose.of.flight" in df_clean.columns:
    df_clean["Purpose.of.flight"] = df_clean["Purpose.of.flight"].str.title()
if "Broad.phase.of.flight" in df_clean.columns:
    df_clean["Broad.phase.of.flight"] = df_clean["Broad.phase.of.flight"].str.title()
if "Weather.Condition" in df_clean.columns:
    df_clean["Weather.Condition"] = df_clean["Weather.Condition"].str.upper()

for col in ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]:
    if col in df_clean.columns:
        print(f"\n{col} (top values):")
        display(df_clean[col].value_counts(dropna=False).head(10))


Engine.Type (top values):


Engine.Type
Reciprocating    12718
<NA>              2276
Turbo Prop         875
Turbo Fan          359
Unknown             56
Turbo Jet           45
Turbo Shaft          8
Name: count, dtype: Int64


Weather.Condition (top values):


Weather.Condition
VMC     13952
<NA>     1378
IMC       866
UNK       141
Name: count, dtype: Int64


Number.of.Engines (top values):


Number.of.Engines
1       13097
2        1918
<NA>     1276
4          27
3          15
0           4
Name: count, dtype: Int64


Purpose.of.flight (top values):


Purpose.of.flight
Personal              9765
Instructional         2382
<NA>                  1741
Aerial Application     720
Business               393
Positioning            253
Unknown                234
Skydiving              156
Aerial Observation     145
Other Work Use         118
Name: count, dtype: Int64


Broad.phase.of.flight (top values):


Broad.phase.of.flight
<NA>           13919
Landing         1105
Takeoff          421
Cruise           233
Approach         209
Maneuvering      126
Taxi              92
Go-Around         81
Descent           59
Climb             48
Name: count, dtype: Int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [20]:
missing_pct = df_clean.isna().mean().sort_values(ascending=False)

# Preserve columns needed for required downstream factor analysis.
protected_cols = {
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight",
}

# Drop only very sparse, non-protected columns.
na_drop_threshold = 0.70
drop_cols = [
    c for c in missing_pct[missing_pct > na_drop_threshold].index.tolist()
    if c not in protected_cols
]

if drop_cols:
    print("Dropping columns with >70% missingness (excluding protected factor columns):")
    print(drop_cols)
    df_clean = df_clean.drop(columns=drop_cols)
else:
    print("No columns exceeded 70% missingness after protected-column check.")

print("\nCurrent shape:", df_clean.shape)
print("\nRemaining missingness (top 15):")
display((df_clean.isna().mean() * 100).sort_values(ascending=False).head(15).round(2))

Dropping columns with >70% missingness (excluding protected factor columns):
['Schedule']

Current shape: (16337, 37)

Remaining missingness (top 15):


Broad.phase.of.flight    85.20
Air.carrier              54.17
Airport.Code             32.27
Airport.Name             31.77
Report.Status            16.88
Engine.Type              13.93
Purpose.of.flight        10.66
Weather.Condition         8.43
Number.of.Engines         7.81
Longitude                 5.83
Latitude                  5.81
Publication.Date          3.13
Aircraft.damage           2.10
FAR.Description           0.95
Registration.Number       0.67
dtype: float64

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [21]:
output_path = "data/aviation_accidents_cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"Saved cleaned data to: {output_path}")
print("Final shape:", df_clean.shape)
print("Final columns:", len(df_clean.columns))

Saved cleaned data to: data/aviation_accidents_cleaned.csv
Final shape: (16337, 37)
Final columns: 37
